# IEEE 9-bus, 4th-order generators: SP vs DP vs EMT

Runs the same IEEE 9-bus system (three 4th-order synchronous machines with AVR and governor, power-flow initialized) in the SP, DP and EMT domains and compares bus voltages and generator behaviour.

The system frequency is selectable with the `-f` argument of each example (default 60 Hz, matching the RTDS source). Set `frequency` below to 50 for the European variant.

In [ ]:
from villas.dataprocessing.readtools import read_timeseries_csv
import matplotlib.pyplot as plt
import numpy as np
import glob
import os
import subprocess
import dpsimpy

frequency = 60

root_path = (
    subprocess.Popen(["git", "rev-parse", "--show-toplevel"], stdout=subprocess.PIPE)
    .communicate()[0]
    .rstrip()
    .decode("utf-8")
)

candidates = [os.path.join(root_path, "build", "dpsim", "examples", "cxx")]
candidates += glob.glob(
    os.path.join(root_path, "build", "*", "dpsim", "examples", "cxx")
)
path_exec = next(
    p for p in candidates if os.path.isfile(os.path.join(p, "DP_Ph1_IEEE9_4Order"))
)
print("binaries:", path_exec)

In [ ]:
names = ["SP_Ph1_IEEE9_4Order", "DP_Ph1_IEEE9_4Order", "EMT_Ph3_IEEE9_4Order"]
for name in names:
    sim = subprocess.Popen(
        [os.path.join(path_exec, name), "-f", str(frequency), "--option", "log=true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    sim.communicate()
    print(name, "done")

In [ ]:
def load(name):
    flat = "logs/" + name + ".csv"
    nested = "logs/" + name + "/" + name + ".csv"
    return read_timeseries_csv(flat if os.path.isfile(flat) else nested)


ts_sp = load("SP_Ph1_IEEE9_4Order")
ts_dp = load("DP_Ph1_IEEE9_4Order")
ts_emt = load("EMT_Ph3_IEEE9_4Order")


# Three-phase magnitude comparable to the SP/DP phasor magnitude:
# for a balanced set sqrt(a^2 + b^2 + c^2) equals the phasor magnitude.
def emt_magnitude(ts, key):
    a, b, c = ts[key + "_0"], ts[key + "_1"], ts[key + "_2"]
    return np.sqrt(a.values**2 + b.values**2 + c.values**2), a.time

## Bus voltages

Voltage magnitude at the 230 kV network buses. The SP and DP phasor magnitudes and the EMT three-phase magnitude coincide.

In [ ]:
plt.figure(figsize=(12, 7))
for bus in ["BUS5", "BUS6", "BUS8"]:
    plt.plot(ts_sp[bus].time, ts_sp[bus].abs().values / 1e3, label=bus + " SP")
    plt.plot(ts_dp[bus].time, ts_dp[bus].abs().values / 1e3, "--", label=bus + " DP")
    mag, t = emt_magnitude(ts_emt, bus)
    plt.plot(t, mag / 1e3, ":", label=bus + " EMT")
plt.xlabel("time [s]")
plt.ylabel("|V| [kV]")
plt.legend()
plt.show()

EMT resolves the instantaneous waveform while SP and DP carry the envelope. Phase a of BUS5 with the DP envelope (phase peak) overlaid.

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(
    ts_emt["BUS5_0"].time, ts_emt["BUS5_0"].values / 1e3, label="BUS5 phase a (EMT)"
)
plt.plot(
    ts_dp["BUS5"].time,
    dpsimpy.RMS3PH_TO_PEAK1PH * ts_dp["BUS5"].abs().values / 1e3,
    "--",
    label="BUS5 envelope (DP)",
)
plt.xlim(0.0, 0.1)
plt.xlabel("time [s]")
plt.ylabel("v [kV]")
plt.legend()
plt.show()

## Generator behaviour

Rotor speed, rotor angle and terminal current magnitude of the three machines across the three domains. The mechanical states (speed, angle) are domain independent and coincide.

In [ ]:
gens = ["GEN1", "GEN2", "GEN3"]

plt.figure(figsize=(12, 7))
for gen in gens:
    plt.plot(
        ts_sp[gen + ".omega"].time, ts_sp[gen + ".omega"].values, label=gen + " SP"
    )
    plt.plot(
        ts_dp[gen + ".omega"].time,
        ts_dp[gen + ".omega"].values,
        "--",
        label=gen + " DP",
    )
    plt.plot(
        ts_emt[gen + ".omega"].time,
        ts_emt[gen + ".omega"].values,
        ":",
        label=gen + " EMT",
    )
plt.xlabel("time [s]")
plt.ylabel("rotor speed [pu]")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
for gen in gens:
    plt.plot(
        ts_sp[gen + ".delta"].time, ts_sp[gen + ".delta"].values, label=gen + " SP"
    )
    plt.plot(
        ts_dp[gen + ".delta"].time,
        ts_dp[gen + ".delta"].values,
        "--",
        label=gen + " DP",
    )
    plt.plot(
        ts_emt[gen + ".delta"].time,
        ts_emt[gen + ".delta"].values,
        ":",
        label=gen + " EMT",
    )
plt.xlabel("time [s]")
plt.ylabel("rotor angle [rad]")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
for gen in gens:
    plt.plot(ts_sp[gen + ".I"].time, ts_sp[gen + ".I"].abs().values, label=gen + " SP")
    plt.plot(
        ts_dp[gen + ".I"].time, ts_dp[gen + ".I"].abs().values, "--", label=gen + " DP"
    )
    mag, t = emt_magnitude(ts_emt, gen + ".I")
    plt.plot(t, mag, ":", label=gen + " EMT")
plt.xlabel("time [s]")
plt.ylabel("terminal current |I| [A]")
plt.legend()
plt.show()

In [ ]:
for bus in ["BUS5", "BUS6", "BUS8"]:
    sp = ts_sp[bus].abs().values.mean()
    dp = ts_dp[bus].abs().values.mean()
    emt = emt_magnitude(ts_emt, bus)[0].mean()
    print(f"{bus}: SP {sp/1e3:.2f} kV  DP {dp/1e3:.2f} kV  EMT {emt/1e3:.2f} kV")
    assert abs(sp - dp) / sp < 0.02
    assert abs(emt - dp) / dp < 0.02